# Deploy an MLflow model with SageMaker

## Install MLflow

In [ ]:
!pip install -q mlflow

## Setup environment

In [ ]:
import json
import boto3
import mlflow
import sagemaker
import pandas as pd
import mlflow.sagemaker
from mlflow.deployments import get_deploy_client

# name of the AWS region to which to deploy the application
region = sagemaker.Session().boto_region_name
# we are using the notebook instance role for training in this example
role = sagemaker.get_execution_role() 
# uri of your remote mlflow server
tracking_uri = 'http://MLflow-MLFLO-JW54W3S07WQL-ddaae86f1ebfd86e.elb.us-east-1.amazonaws.com' 
# set remote mlflow server
mlflow.set_tracking_uri(tracking_uri)

## Build MLflow docker image to serve the model with SageMaker 

### Instructions to Build Your Own MLflow Docker Image to Serve a Model

You can do this on your local Docker daemon.

1. Make sure you have configured AWS credentials on your PC:

    ```sh
    $ aws configure
    AWS Access Key ID [****]: Your-Access-Key-ID
    AWS Secret Access Key [****]: Your-Secret-Access-Key
    Default region name [None]: us-east-1
    Default output format [None]: json
    ```

2. Install Docker Desktop https://www.docker.com/products/docker-desktop/ (ensure your Docker Desktop is up and running before proceeding with the following steps).

3. Install MLflow:

    ```sh
    $ pip install mlflow
    ```

4. Install Boto3:

    ```sh
    $ pip install boto3
    ```

5. Run the following command to build and push the MLflow container. This step will take 20 to 30 minutes to complete and will automatically push the container to Amazon ECR:

    ```sh
    $ mlflow sagemaker build-and-push-container
    ```

Once you have your own Docker image, you can start using it.


In [ ]:
!docker images

In [ ]:
# Build the MLflow serving image from source and push it to YOUR ECR.
# Creates/uses an ECR repo named "mlflow" and tags it with the MLflow version.
# Requires Docker running locally and AWS credentials configured. Takes ~20-30 min.
!mlflow sagemaker build-and-push-container --build --push --container mlflow

In [ ]:
# ECR image built and pushed above (your account, not a third-party registry).
account_id = boto3.client('sts').get_caller_identity()['Account']
image_uri = f'{account_id}.dkr.ecr.{region}.amazonaws.com/mlflow:{mlflow.__version__}'
print(image_uri)

## Deploy a SageMaker endpoint with our scikit-learn model

In [ ]:
endpoint_name = 'boston-housing-mlops'
# The location, in URI format, of the MLflow model to deploy to SageMaker.
model_uri = 'models:/boston-test/3'

In [ ]:
config={
    'execution_role_arn': role,
    'image_url': image_uri,
    'instance_type': 'ml.m5.xlarge',
    'instance_count': 1, 
    'region_name': region
}

client = get_deploy_client("sagemaker")

# client.create_deployment(
#     name=endpoint_name,
#     model_uri=model_uri,
#     flavor='python_function',
#     config=config
# )


#Use update_deployment to update the existing endpoint with the new model version.
client.update_deployment(
    name=endpoint_name,
    model_uri=model_uri,
    flavor='python_function',
    config=config
)

## Predict

In [ ]:
# Hardcoded Boston dataset (sample rows)
boston_data = [
    [0.00632, 18.0, 2.31, 0.0, 0.538, 6.575, 65.2, 4.0900, 1.0, 296.0, 15.3, 396.90, 4.98],
    [0.02731, 0.0, 7.07, 0.0, 0.469, 6.421, 78.9, 4.9671, 2.0, 242.0, 17.8, 396.90, 9.14],
    [0.02729, 0.0, 7.07, 0.0, 0.469, 7.185, 61.1, 4.9671, 2.0, 242.0, 17.8, 392.83, 4.03],
    [0.03237, 0.0, 2.18, 0.0, 0.458, 6.998, 45.8, 6.0622, 3.0, 222.0, 18.7, 394.63, 2.94],
    [0.06905, 0.0, 2.18, 0.0, 0.458, 7.147, 54.2, 6.0622, 3.0, 222.0, 18.7, 396.90, 5.33],
    [0.02985, 0.0, 2.18, 0.0, 0.458, 6.430, 58.7, 6.0622, 3.0, 222.0, 18.7, 394.12, 5.21],  # Row 5
    [0.08829, 12.5, 7.87, 0.0, 0.524, 6.012, 66.6, 5.5605, 5.0, 311.0, 15.2, 395.60, 12.43],
    [0.14455, 12.5, 7.87, 0.0, 0.524, 6.172, 96.1, 5.9505, 5.0, 311.0, 15.2, 396.90, 19.15],
]

# Feature names for the Boston dataset
feature_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 
                 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']

# Create DataFrame
df = pd.DataFrame(boston_data, columns=feature_names)

# Your prediction code
client = get_deploy_client(f"sagemaker:/{region}")
payload = df.iloc[[2]]
prediction = client.predict(endpoint_name, df.iloc[[2]])
print(f'Payload: {payload}')
print(f'Prediction: {prediction}')

## Delete endpoint

In [ ]:
client.delete_deployment('boston-housing-mlops', config=config)